In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install transformer
!pip install datasets
!pip install evaluate

ERROR: Could not find a version that satisfies the requirement transformer (from versions: none)
ERROR: No matching distribution found for transformer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import torch
import pandas as pd 
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset
import evaluate

In [3]:
# distilbert-base-uncased
dataset = load_dataset("SetFit/ag_news")


train.jsonl:   0%|          | 0.00/33.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7600
    })
})

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

label_id  = {0:'World', 1:'Sci/Tech', 2:'Business', 3:'Sports'}
id_label  = {v: k for k, v in label_id.items()}

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
tokenizer

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
tokenized

DatasetDict({
    train: Dataset({
        features: ['label', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [9]:
tokenized = tokenized.rename_column('label', 'labels')

In [10]:
tokenized

DatasetDict({
    train: Dataset({
        features: ['labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

### Model

In [11]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                            num_labels=4, 
                                                            id2label=label_id,
                                                            label2id=id_label)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Training

In [12]:
import os

folder = "/kaggle/working/model_data"

os.makedirs(folder, exist_ok=True)

In [13]:
args = TrainingArguments(
    output_dir=folder,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=True,
    report_to='none',
    
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
accuracy = evaluate.load("accuracy")

def metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=preds, references=eval_pred.label_ids)
    

In [18]:
trainer = Trainer(
    model=model, 
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrics,
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.231213,0.357182,0.939605
2,0.210516,0.360930,0.944737
3,0.147126,0.384501,0.943816


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5625, training_loss=0.21089076266818577, metrics={'train_runtime': 2427.7118, 'train_samples_per_second': 148.288, 'train_steps_per_second': 2.317, 'total_flos': 1.2536598751294464e+16, 'train_loss': 0.21089076266818577, 'epoch': 3.0})

In [20]:
trainer.save_model(folder) 
tokenizer.save_pretrained(folder)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/model_data/tokenizer_config.json',
 '/kaggle/working/model_data/tokenizer.json')

In [24]:
from transformers import pipeline


In [25]:
class NewsClassifier:
    def __init__(self, model_path='/kaggle/working/model_data'):
        self.pipe = pipeline(
            'text-classification',
            model=model_path,
            device=0 if torch.cuda.is_available() else -1,
        )

    def predict(self, text: str) -> dict:
        result = self.pipe(text, truncation=True)[0]
        return {'category': result['label'], 'confidence': result['score']}
        
    def predict_batch(self, texts: list[str]) -> list[dict]:
        results = self.pipe(texts, truncation=True, batch_size=32)
        return [{'category': r['label'], 'confidence': r['score']} for r in results]

In [31]:
if __name__ == '__main__':
    clf = NewsClassifier()
    article = '''
        
        Apple announced record quarterly earnings driven by iPhone sales
        and services growth, beating analyst expectations by 12 percent.
    '''
    print(clf.predict(article))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'category': 'Business', 'confidence': 0.6689720153808594}
